In [1]:
import pandas as pd
import os

raw_path = os.path.join('..', 'data', 'raw')
proc_path = os.path.join('..', 'data', 'processed')
os.makedirs(proc_path, exist_ok=True)

# 1. Pulizia EDGES
print("Processing edges...")
edges = pd.read_csv(os.path.join(raw_path, 'edges.csv'))
# Rinominiamo le colonne Neo4j in standard Python
edges = edges.rename(columns={':START_ID': 'source', ':END_ID': 'target', ':TYPE': 'type'})
edges.to_csv(os.path.join(proc_path, 'edges_cleaned.csv'), index=False)

Processing edges...


In [2]:
import pandas as pd
import os

raw_path = os.path.join('..', 'data', 'raw')
proc_path = os.path.join('..', 'data', 'processed')
os.makedirs(proc_path, exist_ok=True)

# 1. Caricamento del file originale
# Nota: sostituisci 'nodes.csv' con il nome esatto del tuo file nella cartella raw
raw_nodes = pd.read_csv(os.path.join(raw_path, 'nodes.csv'), low_memory=False)

# 2. Pulizia Nomi Colonne (rimuoviamo i suffissi Neo4j)
raw_nodes.columns = [c.split(':')[0].strip() for c in raw_nodes.columns]
raw_nodes.columns = [c.lstrip(':') for c in raw_nodes.columns]

# 3. Definizione delle colonne da mantenere
# Mappiamo le colonne originali a nomi più semplici e puliti
cols_map = {
    'celex_id': 'celex',
    'work_title': 'title',
    'year': 'year',
    'domains': 'domains',
    'subdomains': 'subdomains',
    'resource_legal_type': 'legal_type',
    'url': 'url'
}

# Se 'celex_id' non esiste, cerchiamo 'id' come fallback
if 'celex_id' not in raw_nodes.columns and 'id' in raw_nodes.columns:
    raw_nodes = raw_nodes.rename(columns={'id': 'celex_id'})

# Selezioniamo solo le colonne che esistono effettivamente nel file raw
existing_cols = [c for c in cols_map.keys() if c in raw_nodes.columns]
nodes_cleaned = raw_nodes[existing_cols].rename(columns=cols_map)

# 4. Filtro Nodi spuri (quelli senza CELEX o con ID alfanumerici casuali)
# Teniamo solo le righe dove celex non è nullo e non è la stringa 'N/A'
nodes_cleaned = nodes_cleaned[nodes_cleaned['celex'].notna() & (nodes_cleaned['celex'] != 'N/A')]

# 5. Fix per il Titolo (work_title)
# Se il titolo è vuoto o NaN, usiamo il CELEX come titolo di emergenza
nodes_cleaned['title'] = nodes_cleaned['title'].fillna(nodes_cleaned['celex'])

# 6. Pulizia stringhe EuroVoc (rimozione di parentesi e virgolette residue)
for col in ['domains', 'subdomains']:
    if col in nodes_cleaned.columns:
        nodes_cleaned[col] = nodes_cleaned[col].astype(str).str.replace(r'[\[\]"]', '', regex=True)
        # Se dopo la pulizia la stringa è 'nan', mettiamo il nostro default
        nodes_cleaned[col] = nodes_cleaned[col].replace('nan', '00 UNKNOWN')

# 7. Salvataggio finale
nodes_cleaned.to_csv(os.path.join(proc_path, 'nodes_cleaned.csv'), index=False)
print(f"Dataset normalizzato con successo! {len(nodes_cleaned)} nodi salvati in 'nodes_cleaned.csv'.")

Dataset normalizzato con successo! 86357 nodi salvati in 'nodes_cleaned.csv'.


In [4]:
import pandas as pd
import os

# 1. Definizione percorsi (usiamo i percorsi relativi alla radice del progetto)
raw_path = os.path.join('..', 'data', 'raw')
proc_path = os.path.join('..', 'data', 'processed')
os.makedirs(proc_path, exist_ok=True)

def clean_headers(df):
    """Rimuove i suffissi Neo4j dai nomi delle colonne (es. id:ID -> id)"""
    new_cols = []
    for col in df.columns:
        clean_name = col.split(':')[0] if ':' in col else col
        # Gestisce i casi tipo :LABEL o :TYPE (rimuove il due punti iniziale)
        if not clean_name and ':' in col:
            clean_name = col.split(':')[1]
        new_cols.append(clean_name.lower())
    df.columns = new_cols
    return df

print("--- Inizio Normalizzazione Dataset ---")

# --- A. Pulizia NODES (Gestione NaN definitiva) ---
nodes_file = os.path.join(proc_path, 'nodes_cleaned.csv')
if os.path.exists(nodes_file):
    nodes = pd.read_csv(nodes_file)
    nodes['domains'] = nodes['domains'].fillna('00 UNKNOWN')
    nodes['celex'] = nodes['celex'].fillna('N/A')
    nodes.to_csv(nodes_file, index=False)
    print(f"1. nodes_cleaned.csv: NaN risolti per {len(nodes)} righe.")

# --- B. Pulizia EUROVOC CONCEPTS (Dizionario) ---
concepts_raw = os.path.join(raw_path, 'eurovoc_concept.csv')
if os.path.exists(concepts_raw):
    concepts = pd.read_csv(concepts_raw)
    concepts = clean_headers(concepts)
    # Rimuoviamo le parentesi quadre STRING[] dai dati se presenti
    for col in ['eurovoc_concepts', 'domains', 'subdomains']:
        if col in concepts.columns:
            concepts[col] = concepts[col].str.replace(r'[\[\]"]', '', regex=True)
    concepts.to_csv(os.path.join(proc_path, 'eurovoc_concepts_cleaned.csv'), index=False)
    print("2. eurovoc_concepts_cleaned.csv: Creato (nomi colonne puliti).")

# --- C. Pulizia ARCHI (has_concept e parent_of) ---
for f_name in ['has_concept_edges.csv', 'parent_of.csv']:
    f_path = os.path.join(raw_path, f_name)
    if os.path.exists(f_path):
        df = pd.read_csv(f_path)
        # Rinominiamo i :START_ID e :END_ID in source e target
        df.columns = ['source', 'target', 'type']
        out_name = f_name.replace('.csv', '_cleaned.csv')
        df.to_csv(os.path.join(proc_path, out_name), index=False)
        print(f"3. {out_name}: Creato con colonne standard (source, target).")

print("--- Normalizzazione Completata! ---")

--- Inizio Normalizzazione Dataset ---
1. nodes_cleaned.csv: NaN risolti per 86357 righe.
2. eurovoc_concepts_cleaned.csv: Creato (nomi colonne puliti).
3. has_concept_edges_cleaned.csv: Creato con colonne standard (source, target).
3. parent_of_cleaned.csv: Creato con colonne standard (source, target).
--- Normalizzazione Completata! ---


### Data Augmentation (Arricchimento):

Meta-dati dal CELEX: Estratti automaticamente Anno e Tipo di Atto (Regulation, Directive, etc.) effettuando il parsing della stringa CELEX.

Back-filling Domini: Recuperati i domini EuroVoc mancanti incrociando i nodi con gli archi has_concept_edges e il dizionario eurovoc_concepts.

In [5]:
import pandas as pd
import os

proc_path = os.path.join('..', 'data', 'processed')

# Carichiamo i file necessari
nodes = pd.read_csv(os.path.join(proc_path, 'nodes_cleaned.csv'))
concepts = pd.read_csv(os.path.join(proc_path, 'eurovoc_concepts_cleaned.csv'))
edges_concept = pd.read_csv(os.path.join(proc_path, 'has_concept_edges_cleaned.csv'))

# 1. Parsing del CELEX per Year e Legal Type
def parse_celex(celex):
    if pd.isna(celex) or len(celex) < 7: return None, None
    
    # L'anno è quasi sempre nelle posizioni 1-4 (es. 32019...)
    # Nota: per i trattati (es. 12016...) l'anno è quello della firma o pubblicazione
    year = celex[1:5] 
    
    # La lettera in posizione 6 (indice 5) definisce il tipo di atto
    l_type = celex[5].upper() 
    
    type_map = {
        'R': 'Regulation',             # Regolamento
        'L': 'Directive',              # Direttiva
        'D': 'Decision',               # Decisione
        'C': 'Communication/Notice',    # Comunicazione/Parere
        'E': 'CFSP Act',               # Atti PESC (Foreign & Security Policy)
        'A': 'International Agreement', # Accordi Internazionali / OMC
        'X': 'Other Act',              # Altri atti
        'Q': 'Internal Regulation',    # Regolamenti interni
        'H': 'Recommendation',         # Raccomandazione
        'O': 'ECB Guideline',          # Orientamenti BCE
        'B': 'Budget',                 # Bilancio
    }
    
    # Gestione speciale per i Trattati (Settore 1)
    if celex.startswith('1'):
        return year, 'Treaty'
    
    return year, type_map.get(l_type, 'Other/Secondary')

# Applicazione al DataFrame
nodes[['year', 'legal_type']] = nodes['celex'].apply(lambda x: pd.Series(parse_celex(str(x))))

# 2. Back-filling dei DOMAINS tramite gli archi
# Creiamo una mappa Concept -> Domain dal file eurovoc_concepts
concept_to_domain = concepts.set_index('id')['domains'].to_dict()

# Troviamo i domini per ogni legge tramite gli archi
edges_concept['domain_resolved'] = edges_concept['target'].map(concept_to_domain)

# Raggruppiamo i domini trovati per ogni legge (source)
law_domains = edges_concept.groupby('source')['domain_resolved'].first().to_dict()

# Riempiamo i domini mancanti nei nodi
nodes['domains'] = nodes['domains'].replace('00 UNKNOWN', pd.NA)
nodes['domains'] = nodes['domains'].fillna(nodes['celex'].map(law_domains))
nodes['domains'] = nodes['domains'].fillna('00 UNKNOWN')

# 3. Salvataggio finale arricchito
nodes.to_csv(os.path.join(proc_path, 'nodes_cleaned.csv'), index=False)
print("Dati arricchiti! Year, Type e Domains recuperati dove possibile.")

Dati arricchiti! Year, Type e Domains recuperati dove possibile.


In [11]:
import pandas as pd
import os
import re

# 1. SETUP PERCORSI 
raw_path = os.path.join('..', 'data', 'raw')
proc_path = os.path.join('..', 'data', 'processed')
os.makedirs(proc_path, exist_ok=True) 

# 2. CARICAMENTO DATI
# Assicurati di caricare il file corretto (cambia il nome se necessario)
nodes_cleaned = pd.read_csv(os.path.join(proc_path, 'nodes_cleaned.csv'))

# 3. FUNZIONE DI PULIZIA ROBUSTA
def strip_eurovoc_codes(text):
    # Gestisce i valori nulli (NaN) senza trasformarli in stringhe "nan"
    if pd.isna(text) or str(text).strip().lower() == 'nan' or str(text).strip() == '':
        return "Settore Trasversale" # Assegniamo un nome chiaro ai dati mancanti
    
    # Rimuove codici numerici (es: "20 TRADE" -> "TRADE")
    cleaned = re.sub(r'\d+\s+', '', str(text)).strip()
    return cleaned.upper() # Uniformiamo tutto in maiuscolo

# 4. APPLICAZIONE
nodes_cleaned['domains'] = nodes_cleaned['domains'].apply(strip_eurovoc_codes)
nodes_cleaned['subdomains'] = nodes_cleaned['subdomains'].apply(strip_eurovoc_codes)

# 5. VERIFICA INTELLIGENTE
# Invece di guardare la prima riga (che potrebbe essere NaN), cerchiamo un esempio reale
esempio_valido = nodes_cleaned[nodes_cleaned['domains'] != "SETTORE TRASVERSALE"]['domains'].iloc[0]
print(f" Verifica pulizia su dato reale: {esempio_valido}")

# 6. SALVATAGGIO
nodes_cleaned.to_csv(os.path.join(proc_path, 'nodes_cleaned.csv'), index=False)
print(f" File salvato correttamente in: {proc_path}")

 Verifica pulizia su dato reale: UNKNOWN
 File salvato correttamente in: ..\data\processed


## Edges

In [4]:
import pandas as pd

edges = pd.read_csv("../data/processed/edges_cleaned.csv")
edges["type"].value_counts()

type
CITES                          113874
BASED_ON                        58483
AMENDS                          15539
CORRECTS                        10900
ADOPTS                           7365
REPEALS                          4848
IMPLICITLY_REPEALS               4120
DOES_REPLACEMENT                  954
DEROGATES                         672
EXTENDS_VALIDITY                  654
COMPLETES                         527
DOES_INSERTION                    310
DOES_DELETION                     236
REPLACES                          207
IMPLEMENTS                        172
RELATED_TO                        131
EXTENDS_APPLICATION               100
DOES_REPEAL                        70
PARTIALLY_ADOPTS                   59
INFLUENCES                         27
SUSPENDS                           27
ADDS_TO                            11
INTERPRETES_AUTHORITATIVELY         4
PROPOSES_TO_AMEND                   4
DEFERS_APPLICATION                  3
REESTABLISHES                       2
RELATED

### Hybrid Thematic Weight

Calcolo di un peso finale per ogni arco che combina:
1. **Semantic Hierarchy** — peso basato sul tipo di relazione (AMENDS > BASED_ON > REPEALS > CITES)
2. **Link Specificity (IDF)** — peso inverso basato sulla popolarità del nodo target (leggi "universali" come il TFUE pesano meno)

Formula: `Final_Weight(i, j) = Hierarchy_Weight(T) * IDF(j)`

In [6]:
import numpy as np
import pandas as pd
import networkx as nx

# --- Load data ---
edges = pd.read_csv("../data/processed/edges_cleaned.csv")
nodes = pd.read_csv("../data/processed/nodes_cleaned.csv")

# --- Step 1: Semantic Hierarchy ---
HIERARCHY_WEIGHTS = {
    # Structural
    "AMENDS": 10.0, "REPLACES": 10.0, "DOES_INSERTION": 10.0,
    "DOES_DELETION": 10.0, "COMPLETES": 10.0, "INCORPORATES": 10.0,
    # Genetic
    "BASED_ON": 5.0, "IMPLEMENTS": 5.0, "ADOPTS": 5.0, "REESTABLISHES": 5.0,
    # Succession / Thematic
    "REPEALS": 2.0, "DEROGATES": 2.0, "SUSPENDS": 2.0,
    # Informational (explicit defaults; all others also get 1.0)
    "CITES": 1.0, "RELATED_TO": 1.0,
}

edges["hierarchy_weight"] = edges["type"].map(HIERARCHY_WEIGHTS).fillna(1.0)

# --- Step 2: IDF (Inverse Document Frequency) ---
total_nodes = nodes["celex"].nunique()

G = nx.from_pandas_edgelist(edges, source="source", target="target",
                            create_using=nx.DiGraph())

in_degrees = pd.Series(dict(G.in_degree()), name="in_degree")

edges["target_in_degree"] = edges["target"].map(in_degrees).fillna(0)

EPSILON = 1e-6
edges["idf"] = np.log(total_nodes / (1 + edges["target_in_degree"])) + EPSILON

# --- Step 3: Final Weight ---
edges["final_weight"] = edges["hierarchy_weight"] * edges["idf"]

# --- Save ---
output = edges[["source", "target", "type", "final_weight"]]
output.to_csv("../data/processed/edges_weighted.csv", index=False)

print(f"Total nodes: {total_nodes}")
print(f"Total edges: {len(edges)}")
print(f"\nWeight stats:\n{output['final_weight'].describe()}")
output.head(10)


C:\Users\claud\AppData\Local\Temp\ipykernel_24236\2913795795.py:7: DtypeWarning: Columns (0: year) have mixed types. Specify dtype option on import or set low_memory=False.
  nodes = pd.read_csv("../data/processed/nodes_cleaned.csv")


Total nodes: 86357
Total edges: 219302

Weight stats:
count    219302.000000
mean         22.098607
std          22.258906
min           2.858497
25%           8.034042
50%          10.267634
75%          31.241262
max         106.730990
Name: final_weight, dtype: float64


,source,target,type,final_weight
0,31996D0546,31995M0595,CITES,10.673099
1,31996D0546,11992E086,CITES,7.605046
2,31996D0546,31990L0388,CITES,8.275204
3,31996D0546,31994D0579,CITES,9.574487
4,31996D0546,31994D0896,CITES,10.673099
5,31996D0546,31994D0322,CITES,10.673099
6,31996D0546,31993D0403,CITES,10.673099
7,31996D0546,21994A0103(22),CITES,9.420336
8,31996D0546,31996D0547,CITES,9.756808
9,31996D0546,11992E085,CITES,7.323195
